<a href="https://colab.research.google.com/github/albhoe/593Project/blob/main/generatetagproject.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers datasets accelerate peft

In [ ]:
pip install --upgrade torchao

In [ ]:
import pandas as pd
import torch
from transformers import (
    BartForConditionalGeneration,
    BartTokenizer,
    TrainingArguments,
    Trainer,
    get_scheduler,
    AutoModelForCausalLM
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
import io
import os
from torch.optim import AdamW
import numpy as np
from sklearn.metrics import accuracy_score
from google.colab import drive
import gc

In [ ]:
#Google drive is used to store the dataset as well as saving model weights for
#later use because it is easy to integrate with Colab, which is how this notebook
#was developed. This is particularly useful for transferring models to a
#seperate testing environment without retraining every time.
drive.mount('/content/drive')

In [ ]:
file_path_drive = '/content/drive/MyDrive/593project/ao3_14400001-14500000.jsonl'
save_path_drive = '/content/drive/MyDrive/593project/fine_tuned_bart_lora_generation_saved'

line_count = 0
df = pd.DataFrame()

with open(file_path_drive, 'r', encoding='utf-8') as f:
    for line in f:
        if line_count >= 5000: #This appears to be the limiting factor on training in terms of time.
            break
        line_count += 1
        df = pd.concat([df, pd.read_json(io.StringIO(line), lines=True)], ignore_index=True)
metadata_df = pd.json_normalize(df['metadata'])
df = df.drop(columns=['metadata','id'])
df = pd.concat([df.reset_index(drop=True), metadata_df.reset_index(drop=True)], axis=1)
display(df.head())
#In this generation model, only the input text and output tags are used. Relationship tags
#are the main focus
df = df[['text','Relationship']]

print("\nFirst 5 rows of data from Google Drive:")
display(df.head())

Prepare the models

In [ ]:
#This block is a RAM cleanup step that was used during testing to conserve resources on Colab.
gc.collect()

# If using CUDA (GPU), clear the CUDA cache
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

#Both generatetagproject and labeltagproject use facebook BART as the base model.
#However, this generation task uses the encoder and the decoder together.
encoder_decoder = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

if os.path.exists(save_path_drive):
  encoder_decoder = PeftModel.from_pretrained(encoder_decoder, save_path_drive)
  config_generation = PeftConfig.from_pretrained(save_path_drive)
  base_model_generation = AutoModelForCausalLM.from_pretrained(config_generation.base_model_name_or_path)
  generation_model = PeftModel.from_pretrained(base_model_generation, save_path_drive)

else:
  generation_lora_config = LoraConfig(
      r=8,  # rank
      lora_alpha=8,
      target_modules=["q_proj", "v_proj"],
      lora_dropout=0.1,
      bias="none",
      task_type="SEQ_2_SEQ_LM" #Suited for summary/translation
  )
  generation_peft_model = get_peft_model(encoder_decoder, generation_lora_config)

generation_peft_model.print_trainable_parameters()

Part 2: Generative tags. This will be more difficult, if possible at all. The idea is to lean into BART's attention mechanisms to learn all aspects formatting such as various tropes and patterns and then generate the appropriate tags.

In [ ]:
def preprocess_function(examples):
  #Map relationship tags to their tokenized string forms. Maps main text to tokenized string form.
  additional_tags_processed = [str(tag) if pd.notna(tag) else "" for tag in examples["Relationship"]]

  model_inputs = tokenizer(examples['text'], max_length=1024, truncation=True, padding="max_length")
  model_inputs["labels"] = tokenizer(additional_tags_processed, max_length=1024, truncation=True, padding="max_length").input_ids
  return model_inputs

tag_df = df[['Relationship','text']].copy(deep=True)
display(tag_df.head())

dataset = Dataset.from_pandas(tag_df)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

train_dataset = tokenized_dataset.remove_columns(['text','Relationship']).train_test_split(test_size=0.2, shuffle=True, seed=42)['train']
eval_dataset = tokenized_dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)['test']

In [ ]:
#import torch_xla.core.xla_model as xm

# Detect device
bf16_enabled = False
fp16_enabled = False
"""try:
    # Check for TPU
    #if xm.xla_resource_manager().get_xla_supported_device_type() == 'TPU':
        device = xm.xla_device()
        print(f"Using TPU: {device}")
        bf16_enabled = True # bf16 is often preferred for TPUs
    #else:
        #raise Exception("Not a TPU runtime.")
#except Exception:
    # Fallback to GPU or CPU"""
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {device}")
    fp16_enabled = True # fp16 is often preferred for GPUs
else:
    device = torch.device("cpu")
    print(f"Using CPU: {device}")


training_args = TrainingArguments(
    warmup_steps=10,
    output_dir='./results_generation',
    num_train_epochs=10,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    logging_dir='./logs_generation',
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    bf16=bf16_enabled,
    fp16=fp16_enabled,
    report_to='none',
    learning_rate=5e-5
)

generation_peft_model.to(device) # Move model to the detected device

optimizer = AdamW(generation_peft_model.parameters(), lr=training_args.learning_rate)

total_train_steps = int(len(train_dataset) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps) * training_args.num_train_epochs)

# Define a learning rate scheduler
lr_scheduler = get_scheduler(
    name=training_args.lr_scheduler_type, # Defaults to 'linear'
    optimizer=optimizer,
    num_warmup_steps=training_args.warmup_steps,
    num_training_steps=total_train_steps,
)

generation_peft_model.print_trainable_parameters()


trainer = Trainer(
    model=generation_peft_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    optimizers=(optimizer, lr_scheduler) # Pass the custom optimizer and scheduler
)

trainer.train()

os.makedirs(save_path_drive, exist_ok=True)
generation_peft_model.to('cpu').save_pretrained(save_path_drive)

In [ ]:
display(eval_dataset)

In [ ]:
generation_peft_model.eval()
generation_peft_model.to(device) # Ensure model is on the correct device for evaluation
sample_indices = range(min(10, len(eval_dataset))) # Display up to 10 samples, or fewer if eval_dataset is smaller

print("\nSample of Generated vs. True Relationship Tags:")
print("=================================================")

for i in sample_indices:
    input_text = eval_dataset['text'][i]
    true_tags = eval_dataset["Relationship"][i]

    # Prepare input for generation
    inputs = tokenizer(input_text, return_tensors='pt', max_length=1024, truncation=True).to(device)

    # Create decoder_input_ids to force the decoder to start generating tags
    # The decoder_start_token_id is usually the EOS token for BART models.
    decoder_start_token_id = generation_peft_model.config.decoder_start_token_id
    decoder_input_ids = torch.ones((1, 1), dtype=torch.long, device=device) * decoder_start_token_id

    # Generate output
    with torch.no_grad():
        generated_ids = generation_peft_model.generate(
            input_ids=inputs.input_ids,
            decoder_input_ids=decoder_input_ids, # Explicitly provide decoder start tokens
            max_new_tokens=64, # Max length for generated tags
            num_beams=2,        # Beam search for better quality
            early_stopping=True # Stop when all beam hypotheses have finished
        )

    # Decode generated tags
    generated_tags = tokenizer.decode(generated_ids[0], skip_special_tokens=True)

    print(f"\n--- Sample {i+1} ---")
    print(f"Original Text: {input_text[:200]}...") # Truncate for display
    print(f"True Tags:     {true_tags}")
    print(f"Generated Tags: {generated_tags}")

In [ ]:
def label_accuracy(string1:str,string2:str):
  set1 = set(string1.split(', '))
  set2 = set(string2.split(', '))

  difference1 = set1.difference(set2)
  difference2 = set2.difference(set1)
  intersection = set1.intersection(set2)

  return (difference1,intersection,difference2)

label_accuracy("cheese, crackers, pasta, meatballs","meatballs, cheese, tomatoes")